In [5]:
# before running the version-specific tests, a small set of helper functions is defined
# these helpers are reused across all notebooks
import time
import numpy as np
import pandas as pd


# this function measures the average execution time of a given function
# used to test inference speed and overall performance
def measure_runtime(func, runs=5):
    times = []
    for _ in range(runs):
        start = time.time()
        func()
        times.append(time.time() - start)
    return np.mean(times)


# the intersection over union measures the overlap between two bounding boxes
# used to evaluate spatial consistency between detections
def iou(bb1, bb2):
    x1 = max(bb1[0], bb2[0])
    y1 = max(bb1[1], bb2[1])
    x2 = min(bb1[2], bb2[2])
    y2 = min(bb1[3], bb2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (bb1[2] - bb1[0]) * (bb1[3] - bb1[1])
    area2 = (bb2[2] - bb2[0]) * (bb2[3] - bb2[1])

    return inter / (area1 + area2 - inter + 1e-6)

In [ ]:
# speed / runtime performance
# this test measures how long it takes to run inference on a single image
def v1_inference():
    model.predict(source=image_path, imgsz=640, conf=0.25)

avg_time = measure_runtime(v1_inference)
print(f"[V1] Average inference time per image: {avg_time:.3f} sec")

In [ ]:
# detection accuracy
# this test compares the predicted people count to a manually counted ground truth value
GROUND_TRUTH = 5
predicted = count_people

accuracy = 1 - abs(predicted - GROUND_TRUTH) / GROUND_TRUTH
print(f"[V1] Detection accuracy (approx): {accuracy:.2f}")

In [ ]:
# overlap stability
# this test checks how consistent bounding boxes locations are within a single image
ious = []
for i in range(len(person_boxes) - 1):
    ious.append(iou(person_boxes[i], person_boxes[i+1]))

mean_iou = np.mean(ious) if ious else 1.0
print(f"[V1] Mean IoU between detections: {mean_iou:.2f}")

In [ ]:
# determinism test
# this test verifies whether repeated inference on the same image produces consistent results
counts = []
for _ in range(3):
    res = model.predict(image_path)[0]
    classes = res.boxes.cls.cpu().numpy().astype(int)
    counts.append(np.sum(classes == 0))

print(f"[V1] Repeated counts: {counts}")